In [8]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transforms.ToTensor())
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256)

class MLPClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )
    def forward(self, x):
        return self.net(x)

model = MLPClassifier().to(device)
epochs = 5
criterion = nn.CrossEntropyLoss()
optim = torch.optim.SGD(model.parameters(), lr=0.1)
losses = []

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)
        pred = model(x)
        loss = criterion(pred, y)
        optim.zero_grad()
        loss.backward()
        optim.step()
        total_loss += loss.item()
    print(f'epoch: {epoch}, loss: {total_loss / len(train_loader)}')

model.eval()
correct=total=0

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        y = y.to(device)
        pred = model(x)
        predicted = pred.argmax(dim=1)
        correct += (predicted == y).sum().item()
        total += y.size(0)
print(f'test_acc: {correct / total}')
# torch.cuda.is_available()

epoch: 0, loss: 0.5684188311097464
epoch: 1, loss: 0.28496974709827
epoch: 2, loss: 0.2329143594735975
epoch: 3, loss: 0.19471551502532541
epoch: 4, loss: 0.16641390809753556
test_acc: 0.9553
